## Extração de dados da SPTrans entre 2015 a 2026

Este notebook realiza o processo de extração de dados do site da SPTrans, que é armazenada em arquivos xls/xlsx com informações diárias. É realizado um webscraping para pegar todos os links de cada xls, feito o download e consolidação dos dados em um único arquivo parquet. 

In [1]:
import os
import re
import unicodedata
from html import unescape
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests

ROOT = os.path.abspath(os.getcwd())
if os.path.basename(ROOT) == "notebooks":
    ROOT = os.path.dirname(ROOT)

DADOS_DIR = os.path.join(ROOT, "dados")
RAW_DIR = os.path.join(DADOS_DIR, "raw")
os.makedirs(RAW_DIR, exist_ok=True)

PAGINAS_POR_ANO = {
    2026: "https://prefeitura.sp.gov.br/web/mobilidade/w/passageiros-transportados-2026",
    2025: "https://prefeitura.sp.gov.br/web/mobilidade/w/passageiros-transportados-2025",
    2024: "https://capital.sp.gov.br/web/mobilidade/w/institucional/sptrans/acesso_a_informacao/362878",
    2023: "https://prefeitura.sp.gov.br/web/mobilidade/w/institucional/sptrans/acesso_a_informacao/343693",
    2022: "https://prefeitura.sp.gov.br/web/mobilidade/w/institucional/sptrans/acesso_a_informacao/322860",
    2021: "https://prefeitura.sp.gov.br/web/mobilidade/w/institucional/sptrans/acesso_a_informacao/306932",
    2020: "https://www.prefeitura.sp.gov.br/cidade/secretarias/transportes/institucional/sptrans/acesso_a_informacao/index.php?p=292723",
    2019: "https://www.prefeitura.sp.gov.br/cidade/secretarias/transportes/institucional/sptrans/acesso_a_informacao/index.php?p=269652",
    2018: "https://www.prefeitura.sp.gov.br/cidade/secretarias/transportes/institucional/sptrans/acesso_a_informacao/index.php?p=247850",
    2017: "https://www.prefeitura.sp.gov.br/cidade/secretarias/transportes/institucional/sptrans/acesso_a_informacao/index.php?p=228269",
    2016: "https://www.prefeitura.sp.gov.br/cidade/secretarias/transportes/institucional/sptrans/acesso_a_informacao/index.php?p=209427",
    2015: "https://www.prefeitura.sp.gov.br/cidade/secretarias/transportes/institucional/sptrans/acesso_a_informacao/index.php?p=188767",
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/127.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",
    "Connection": "keep-alive",
}

session = requests.Session()
session.headers.update(HEADERS)

In [2]:
def baixar_html(url: str) -> str:
    response = session.get(url, timeout=30)
    response.raise_for_status()
    return response.text


def extrair_anchors(html: str, page_url: str) -> list[tuple[str, str]]:
    padrao = re.compile(r"<a\b[^>]*href=[\"']([^\"']+)[\"'][^>]*>(.*?)</a>", flags=re.IGNORECASE | re.DOTALL)
    anchors = []
    for href, texto_html in padrao.findall(html):
        texto_limpo = re.sub(r"<[^>]+>", "", texto_html)
        texto_limpo = unescape(re.sub(r"\s+", " ", texto_limpo)).strip()
        anchors.append((texto_limpo, urljoin(page_url, href)))
    return anchors


def eh_link_planilha(href_base: str) -> bool:
    base = href_base.lower()
    return base.endswith(".xls") or base.endswith(".xlsx") or base.endswith("-xls") or base.endswith("-xlsx")


def extrair_links_xls(html: str, page_url: str) -> list[str]:
    vistos = set()
    resultado = []

    for texto, href in extrair_anchors(html, page_url):
        href_limpo = href.strip()
        href_base = href_limpo.split("?", 1)[0]
        if not eh_link_planilha(href_base):
            continue

        texto_lower = texto.lower()
        nome_arquivo = href_base.rsplit("/", 1)[-1].lower()

        if "todos" in texto_lower:
            continue
        if "conslidado" in nome_arquivo or "consolidado" in nome_arquivo or "consolidade" in nome_arquivo:
            continue
        if "pass_transp" in nome_arquivo:
            continue

        if href_limpo not in vistos:
            vistos.add(href_limpo)
            resultado.append(href_limpo)

    return resultado


linhas = []
for ano, pagina in sorted(PAGINAS_POR_ANO.items()):
    try:
        html = baixar_html(pagina)
        links = extrair_links_xls(html, pagina)
        if not links:
            linhas.append({"ano": ano, "pagina": pagina, "link_xls": None})
        else:
            for link in links:
                linhas.append({"ano": ano, "pagina": pagina, "link_xls": link})
    except Exception as exc:
        linhas.append({"ano": ano, "pagina": pagina, "link_xls": f"ERRO: {exc}"})


df_links = pd.DataFrame(linhas)
df_xls = (
    df_links[
        df_links["link_xls"].notna()
        & ~df_links["link_xls"].str.startswith("ERRO:", na=False)
    ]
    .drop_duplicates(subset=["ano", "link_xls"])
    .sort_values(["ano", "link_xls"])
    .reset_index(drop=True)
)

df_xls


,ano,pagina,link_xls
0,2015,https://www.prefeitura.sp.gov.br/cidade/secret...,http://www.prefeitura.sp.gov.br/cidade/secreta...
1,2015,https://www.prefeitura.sp.gov.br/cidade/secret...,http://www.prefeitura.sp.gov.br/cidade/secreta...
2,2015,https://www.prefeitura.sp.gov.br/cidade/secret...,http://www.prefeitura.sp.gov.br/cidade/secreta...
3,2015,https://www.prefeitura.sp.gov.br/cidade/secret...,http://www.prefeitura.sp.gov.br/cidade/secreta...
4,2015,https://www.prefeitura.sp.gov.br/cidade/secret...,http://www.prefeitura.sp.gov.br/cidade/secreta...
...,...,...,...
4236,2026,https://prefeitura.sp.gov.br/web/mobilidade/w/...,https://prefeitura.sp.gov.br/documents/d/mobil...
4237,2026,https://prefeitura.sp.gov.br/web/mobilidade/w/...,https://prefeitura.sp.gov.br/documents/d/mobil...
4238,2026,https://prefeitura.sp.gov.br/web/mobilidade/w/...,https://prefeitura.sp.gov.br/documents/d/mobil...
4239,2026,https://prefeitura.sp.gov.br/web/mobilidade/w/...,https://prefeitura.sp.gov.br/documents/d/mobil...


In [3]:
def normalizar_url_xls(url: str) -> str:
    u = url.strip()
    u = re.sub(r"^(https?://)\1", r"\1", u, flags=re.IGNORECASE)
    u = re.sub(r"^(https?://)(https?://)", r"\2", u, flags=re.IGNORECASE)

    marker = "www.prefeitura.sp.gov.br/cidade/secretarias/upload/"
    if marker in u and "//" in u:
        prefixo, sufixo = u.split(marker, 1)
        if "capital.sp.gov.br/web/mobilidade/w/institucional/sptrans/acesso_a_informacao/" in prefixo:
            u = "https://" + marker + sufixo

    if not re.match(r"^https?://", u, flags=re.IGNORECASE):
        u = "https://" + u.lstrip("/")

    return u


baixados = 0
erros = 0
pulados = 0

for idx, row in df_xls.iterrows():
    url_original = row["link_xls"]
    url = normalizar_url_xls(url_original)
    ano = int(row["ano"])

    nome_base = os.path.basename(urlparse(url).path)
    if not nome_base:
        nome_base = f"arquivo_{idx + 1}.xls"

    nome_arquivo = f"{ano}_{nome_base}"
    destino = os.path.join(RAW_DIR, nome_arquivo)

    if os.path.exists(destino):
        pulados += 1
        continue

    try:
        resposta = session.get(url, timeout=60)
        resposta.raise_for_status()
        with open(destino, "wb") as f:
            f.write(resposta.content)
        baixados += 1
    except Exception as exc:
        erros += 1
        print(f"ERRO ao baixar {url_original} (normalizado: {url}): {exc}")

print(f"Baixados: {baixados}")
print(f"Já existiam (pulados): {pulados}")
print(f"Erros: {erros}")
print(f"Diretório de saída: {os.path.abspath(RAW_DIR)}")


Baixados: 0
Já existiam (pulados): 4241
Erros: 0
Diretório de saída: c:\Src\projeto-aplicado-iv\dados\raw


In [4]:
def limpar_texto(valor) -> str:
    if pd.isna(valor):
        return ""
    txt = str(valor).replace("\n", " ").replace("\r", " ").strip()
    txt = re.sub(r"\s+", " ", txt)
    return txt


def normalizar_nome_coluna(nome: str) -> str:
    base = limpar_texto(nome).lower()
    base = "".join(
        ch for ch in unicodedata.normalize("NFKD", base) if not unicodedata.combining(ch)
    )
    base = re.sub(r"[^a-z0-9]+", " ", base)
    return re.sub(r"\s+", " ", base).strip()


def ler_planilha_bruta(caminho: str) -> tuple[pd.DataFrame | None, str | None]:
    erros = []
    for engine in ("openpyxl", "xlrd", None):
        try:
            if engine is None:
                return pd.read_excel(caminho, header=None, dtype=str), None
            return pd.read_excel(caminho, header=None, dtype=str, engine=engine), None
        except Exception as exc:
            nome_engine = "engine_auto" if engine is None else engine
            erros.append(f"{nome_engine}: {exc}")
    return None, " | ".join(erros)


def linha_tem_colunas_suficientes(
    df: pd.DataFrame,
    idx_linha: int,
    idx_coluna_inicio: int,
    minimo: int = 4,
) -> bool:
    linha = [limpar_texto(v) for v in df.iloc[idx_linha].tolist()][idx_coluna_inicio:]
    preenchidas = sum(1 for v in linha if v != "")
    return preenchidas >= minimo


def localizar_cabecalho(
    df: pd.DataFrame,
    limite_linhas: int = 120,
    limite_colunas: int = 8,
) -> tuple[int, int] | None:
    if df.empty or df.shape[1] == 0:
        return None

    max_linhas = min(limite_linhas, len(df))
    max_colunas = min(limite_colunas, df.shape[1])

    for idx_linha in range(max_linhas):
        for idx_coluna in range(max_colunas):
            valor = normalizar_nome_coluna(df.iat[idx_linha, idx_coluna])
            if valor == "data" and linha_tem_colunas_suficientes(df, idx_linha, idx_coluna):
                return idx_linha, idx_coluna

    for idx_linha in range(max_linhas):
        for idx_coluna in range(max_colunas):
            valor = normalizar_nome_coluna(df.iat[idx_linha, idx_coluna])
            if valor.startswith("data") and linha_tem_colunas_suficientes(df, idx_linha, idx_coluna):
                return idx_linha, idx_coluna

    return None


def extrair_nomes_colunas(df: pd.DataFrame, idx_linha: int, idx_coluna_inicio: int) -> list[str]:
    linha = [limpar_texto(v) for v in df.iloc[idx_linha].tolist()]
    linha = linha[idx_coluna_inicio:]

    ult_coluna_preenchida = -1
    for i, valor in enumerate(linha):
        if valor != "":
            ult_coluna_preenchida = i

    if ult_coluna_preenchida == -1:
        return []

    return linha[: ult_coluna_preenchida + 1]


arquivos = sorted([os.path.join(RAW_DIR, f) for f in os.listdir(RAW_DIR) if os.path.isfile(os.path.join(RAW_DIR, f))])
print(f"Arquivos encontrados em {os.path.abspath(RAW_DIR)}: {len(arquivos)}")

registros_colunas = []
registros_problemas = []

for arquivo in arquivos:
    df_bruto, erro_leitura = ler_planilha_bruta(arquivo)
    if erro_leitura:
        registros_problemas.append(
            {"arquivo": os.path.basename(arquivo), "motivo": "erro_leitura", "detalhe": erro_leitura}
        )
        continue

    achado = localizar_cabecalho(df_bruto)
    if achado is None:
        registros_problemas.append(
            {
                "arquivo": os.path.basename(arquivo),
                "motivo": "cabecalho_nao_encontrado",
                "detalhe": "Sem linha de cabeçalho com 'Data' + colunas suficientes",
            }
        )
        continue

    idx_linha, idx_coluna = achado
    colunas = extrair_nomes_colunas(df_bruto, idx_linha, idx_coluna)
    if not colunas:
        registros_problemas.append(
            {
                "arquivo": os.path.basename(arquivo),
                "motivo": "linha_cabecalho_vazia",
                "detalhe": f"Cabeçalho em linha {idx_linha + 1}, coluna {idx_coluna + 1}, sem nomes.",
            }
        )
        continue

    for pos, nome_coluna in enumerate(colunas, start=1):
        registros_colunas.append(
            {
                "arquivo": os.path.basename(arquivo),
                "linha_cabecalho": idx_linha + 1,
                "coluna_inicio": idx_coluna + 1,
                "posicao": pos,
                "nome_coluna": nome_coluna,
                "nome_canonico": normalizar_nome_coluna(nome_coluna),
            }
        )


df_colunas = pd.DataFrame(registros_colunas)
if df_colunas.empty:
    print("Nenhuma coluna foi extraída dos arquivos.")
else:
    df_variacoes = (
        df_colunas.groupby(["posicao", "nome_coluna"], as_index=False)
        .agg(
            qtd_arquivos=("arquivo", "nunique"),
            menor_linha_cabecalho=("linha_cabecalho", "min"),
            menor_coluna_inicio=("coluna_inicio", "min"),
        )
        .sort_values(["posicao", "qtd_arquivos", "nome_coluna"], ascending=[True, False, True])
        .reset_index(drop=True)
    )

    df_canonico = (
        df_colunas.groupby(["posicao", "nome_canonico"], as_index=False)
        .agg(
            qtd_arquivos=("arquivo", "nunique"),
            variacoes_originais=("nome_coluna", lambda s: " | ".join(sorted(set(s)))),
        )
        .sort_values(["posicao", "qtd_arquivos", "nome_canonico"], ascending=[True, False, True])
        .reset_index(drop=True)
    )

    print(f"Total de arquivos com cabeçalho detectado: {df_colunas['arquivo'].nunique()}")
    print(f"Total de arquivos com problema: {len(registros_problemas)}")
    print("Distribuição da coluna de início do cabeçalho (A=1):")
    display(
        df_colunas[["arquivo", "coluna_inicio"]]
        .drop_duplicates()
        .groupby("coluna_inicio", as_index=False)
        .size()
        .rename(columns={"size": "qtd_arquivos"})
        .sort_values("coluna_inicio")
        .reset_index(drop=True)
    )

    display(df_variacoes.head(50))
    display(df_canonico.head(50))

    if registros_problemas:
        display(pd.DataFrame(registros_problemas).head(30))


Arquivos encontrados em c:\Src\projeto-aplicado-iv\dados\raw: 4241
WARNING *** file size (442891) not 512 + multiple of sector size (512)
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
Total de arquivos com cabeçalho detectado: 4184
Total de arquivos com problema: 57
Distribuição da coluna de início do cabeçalho (A=1):


,coluna_inicio,qtd_arquivos
0,1,4184


,posicao,nome_coluna,qtd_arquivos,menor_linha_cabecalho,menor_coluna_inicio
0,1,Data,3185,1,1
1,1,DATA,999,1,1
2,2,Grupo,1833,1,1
3,2,Tipo,1352,1,1
4,2,TIPO,999,1,1
5,3,Lote,1833,1,1
6,3,Area,1352,1,1
7,3,AREA,999,1,1
8,4,Empresa,3185,1,1
9,4,EMPRESA,999,1,1


,posicao,nome_canonico,qtd_arquivos,variacoes_originais
0,1,data,4184,DATA | Data
1,2,tipo,2351,TIPO | Tipo
2,2,grupo,1833,Grupo
3,3,area,2351,AREA | Area
4,3,lote,1833,Lote
5,4,empresa,4184,EMPRESA | Empresa
6,5,linha,4184,LINHA | Linha
7,6,passageiros pagtes em dinheiro,3185,Passageiros Pagtes Em Dinheiro
8,6,passageiros pagantes em dinheiro a,999,PASSAGEIROS PAGANTES EM DINHEIRO (A)
9,7,passageiros comum e vt,2352,Passageiros Comum E Vt | Passageiros Comum e VT


,arquivo,motivo,detalhe
0,2018_01062018.xls,erro_leitura,openpyxl: File is not a zip file | xlrd: | en...
1,2018_01092018.xls,erro_leitura,openpyxl: File is not a zip file | xlrd: | en...
2,2018_03122018_Dados para Transparência.xls,erro_leitura,openpyxl: File is not a zip file | xlrd: | en...
3,2018_06062018.xls,erro_leitura,openpyxl: File is not a zip file | xlrd: | en...
4,2018_07062018.xls,erro_leitura,openpyxl: File is not a zip file | xlrd: | en...
5,2018_08062018.xls,erro_leitura,openpyxl: File is not a zip file | xlrd: | en...
6,2018_10112018_Dados para Transparência.xls,erro_leitura,openpyxl: File is not a zip file | xlrd: | en...
7,2018_12112018_Dados para Transparência.xls,erro_leitura,openpyxl: File is not a zip file | xlrd: | en...
8,2018_14092018.xls,erro_leitura,openpyxl: File is not a zip file | xlrd: | en...
9,2018_15102018_Dados para Transparência.xls,erro_leitura,openpyxl: File is not a zip file | xlrd: | en...


In [5]:
def limpar_texto(valor) -> str:
    if pd.isna(valor):
        return ""
    txt = str(valor).replace("\n", " ").replace("\r", " ").strip()
    txt = re.sub(r"\s+", " ", txt)
    return txt


def normalizar_nome(nome: str) -> str:
    base = limpar_texto(nome).lower()
    base = "".join(ch for ch in unicodedata.normalize("NFKD", base) if not unicodedata.combining(ch))
    base = re.sub(r"[^a-z0-9]+", " ", base)
    return re.sub(r"\s+", " ", base).strip()


def ler_planilha_bruta(caminho: str) -> tuple[pd.DataFrame | None, str | None]:
    erros = []
    for engine in ("openpyxl", "xlrd", None):
        try:
            if engine is None:
                return pd.read_excel(caminho, header=None, dtype=str), None
            return pd.read_excel(caminho, header=None, dtype=str, engine=engine), None
        except Exception as exc:
            nome_engine = "engine_auto" if engine is None else engine
            erros.append(f"{nome_engine}: {exc}")
    return None, " | ".join(erros)


def linha_tem_colunas_suficientes(df: pd.DataFrame, idx_linha: int, idx_coluna_inicio: int, minimo: int = 4) -> bool:
    linha = [limpar_texto(v) for v in df.iloc[idx_linha].tolist()][idx_coluna_inicio:]
    preenchidas = sum(1 for v in linha if v != "")
    return preenchidas >= minimo


def localizar_cabecalho(df: pd.DataFrame, limite_linhas: int = 120, limite_colunas: int = 8) -> tuple[int, int] | None:
    if df.empty or df.shape[1] == 0:
        return None

    max_linhas = min(limite_linhas, len(df))
    max_colunas = min(limite_colunas, df.shape[1])

    for idx_linha in range(max_linhas):
        for idx_coluna in range(max_colunas):
            valor = normalizar_nome(df.iat[idx_linha, idx_coluna])
            if valor == "data" and linha_tem_colunas_suficientes(df, idx_linha, idx_coluna):
                return idx_linha, idx_coluna

    for idx_linha in range(max_linhas):
        for idx_coluna in range(max_colunas):
            valor = normalizar_nome(df.iat[idx_linha, idx_coluna])
            if valor.startswith("data") and linha_tem_colunas_suficientes(df, idx_linha, idx_coluna):
                return idx_linha, idx_coluna

    return None


def extrair_colunas(df_bruto: pd.DataFrame, idx_linha: int, idx_coluna_inicio: int) -> list[str]:
    linha = [limpar_texto(v) for v in df_bruto.iloc[idx_linha].tolist()][idx_coluna_inicio:]

    ult_coluna = -1
    for i, valor in enumerate(linha):
        if valor != "":
            ult_coluna = i

    if ult_coluna == -1:
        return []
    return linha[: ult_coluna + 1]


def consolidar_colunas_duplicadas(df: pd.DataFrame) -> pd.DataFrame:
    novo = pd.DataFrame(index=df.index)
    nomes = list(df.columns)

    for nome in list(dict.fromkeys(nomes)):
        idxs = [i for i, n in enumerate(nomes) if n == nome]
        bloco = df.iloc[:, idxs]
        if bloco.shape[1] == 1:
            novo[nome] = bloco.iloc[:, 0]
        else:
            novo[nome] = bloco.bfill(axis=1).iloc[:, 0]

    return novo


de_para_colunas = {
    "data": ["Data"],
    "tipo": ["tipo", "Grupo"],
    "area": ["Area", "Lote"],
    "empresa": ["Empresa"],
    "linha": ["linha"],
    "passageiros_pagantes_em_dinheiro": ["Passageiros Pagtes Em Dinheiro", "PASSAGEIROS PAGANTES EM DINHEIRO (A)"],
    "passageiros_comum_vt": ["Passageiros Comum e VT", "PASSAGEIROS PAGANTES BILHETE ÚNICO COMUM (B)", "Passageiros Pagtes Comum", "Passageiros Comum E Vt"],
    "passageiros_bilhete_mensal": ["Passageiros Pgts Bu Comum M", "PASSAGEIROS PAGANTES BILHETE ÚNICO COMUM MENSAL (C)"],
    "passageiro_estudante": ["Passageiros Pagtes Estudante", "PASSAGEIROS PAGANTES BILHETE ÚNICO ESTUDANTE (D)"],
    "passageiro_estudante_mensal": ["Passageiros Pgts Bu Est Mensal", "PASSAGEIROS PAGANTES BILHETE ÚNICO ESTUDANTE MENSAL (E)"],
    "passageiros_vale_transporte": ["Passageiros Pgts Bu Vt Mensal", "PASSAGEIROS PAGANTES BILHETE ÚNICO VALE TRANSPORTE (F)", "Passageiros Pagtes Bu Vt"],
    "passageiros_pagantes": ["Passageiros Pagantes"],
    "passageiros_vt_mensal": ["PASSAGEIROS PAGANTES BILHETE ÚNICO VALE TRANSPORTE MENSAL (G)", "Passageiros Pgts Bu Vt Mensal"],
    "passageiros_integracao_metro": ["PASSAGEIROS PAGANTES INTEGRAÇÕES METRÔ E CPTM (H)", "Passageiros Pagtes Int M/Cptm"],
    "passageiros_integracao_onibus": ["Passageiros Int Ônibus->Ônibus", "PASSAGEIROS INTEGRAÇÕES ÔNIBUS -> ÔNIBUS (K)", "Passageiros Int Ônibus->Ônibus"],
    "passageiros_gratuidade": ["Passageiros Com Gratuidade", "Passageiros Com Gratuidade Est", "PASSAGEIROS COM GRATUIDADE (L)", "PASSAGEIROS COM GRATUIDADE ESTUDANTE (M)", "PASSAGEIROS COM GRATUIDADE ESTUDANTE (N)"],
    "passageiros_metro_mensal": ["PASSAGEIROS PAGANTES INTEGRAÇÕES METRÔ E CPTM MENSAL (I)"],
    "total_pagantes": ["PASSAGEIROS PAGANTES (J) = (A+B+C+D+E+F+G+H+I)", "TOTAL PASSAGEIROS TRANSPORTADOS (M) = (J + K + L)", "Tot Passageiros Transportados", "TOTAL PASSAGEIROS TRANSPORTADOS (N) = (J + K + L + M)", "TOTAL PASSAGEIROS TRANSPORTADOS (M) = (J + K + L+N)"],
}

mapa_equivalencias = {}
for nome_final, equivalentes in de_para_colunas.items():
    nome_final_norm = normalizar_nome(nome_final)
    mapa_equivalencias[nome_final_norm] = nome_final
    for eq in equivalentes:
        mapa_equivalencias[normalizar_nome(eq)] = nome_final

arquivos = sorted([os.path.join(RAW_DIR, f) for f in os.listdir(RAW_DIR) if os.path.isfile(os.path.join(RAW_DIR, f))])
frames = []
erros_unificacao = []

for arquivo in arquivos:
    df_bruto, erro_leitura = ler_planilha_bruta(arquivo)
    if erro_leitura:
        erros_unificacao.append({"arquivo": os.path.basename(arquivo), "etapa": "leitura", "erro": erro_leitura})
        continue

    cab = localizar_cabecalho(df_bruto)
    if cab is None:
        erros_unificacao.append({"arquivo": os.path.basename(arquivo), "etapa": "cabecalho", "erro": "Linha de cabeçalho não encontrada (Data)"})
        continue

    idx_linha, idx_coluna = cab
    nomes_colunas_orig = extrair_colunas(df_bruto, idx_linha, idx_coluna)
    if not nomes_colunas_orig:
        erros_unificacao.append({"arquivo": os.path.basename(arquivo), "etapa": "cabecalho", "erro": "Sem nomes de coluna na linha detectada"})
        continue

    df_dados = df_bruto.iloc[idx_linha + 1 :, idx_coluna : idx_coluna + len(nomes_colunas_orig)].copy()
    if df_dados.empty:
        continue

    nomes_limpos = [limpar_texto(c) for c in nomes_colunas_orig]
    nomes_norm = [normalizar_nome(c) for c in nomes_limpos]
    df_dados.columns = nomes_limpos

    rename_map = {}
    for original, norm in zip(nomes_limpos, nomes_norm):
        destino = mapa_equivalencias.get(norm)
        if destino:
            rename_map[original] = destino

    df_dados = df_dados.rename(columns=rename_map)
    df_dados = consolidar_colunas_duplicadas(df_dados)

    cols_para_remover = []
    for col in df_dados.columns:
        serie = df_dados[col]
        if serie.astype(str).str.strip().replace("nan", "").eq("").all():
            cols_para_remover.append(col)
    if cols_para_remover:
        df_dados = df_dados.drop(columns=cols_para_remover)

    for col_final in de_para_colunas.keys():
        if col_final not in df_dados.columns:
            df_dados[col_final] = pd.NA

    cols_finais = list(de_para_colunas.keys())
    df_dados = df_dados[cols_finais].copy()

    nome_arquivo = os.path.basename(arquivo)
    ano_match = re.match(r"^(\d{4})_", nome_arquivo)
    df_dados["ano_arquivo"] = int(ano_match.group(1)) if ano_match else pd.NA
    df_dados["arquivo_origem"] = nome_arquivo
    df_dados = df_dados.dropna(how="all", subset=cols_finais)

    if not df_dados.empty:
        frames.append(df_dados)

if frames:
    df_final_unificado = pd.concat(frames, ignore_index=True)
    
    
    print(f"Arquivos processados: {len(arquivos)}")
    print(f"Arquivos com dados aproveitados: {len(frames)}")
    print(f"Linhas no dataframe final: {len(df_final_unificado)}")
    

    if erros_unificacao:
        print(f"Arquivos com erro/sem cabeçalho: {len(erros_unificacao)}")
        display(pd.DataFrame(erros_unificacao).head(20))
else:
    print("Nenhum dataframe válido foi gerado.")
    if erros_unificacao:
        display(pd.DataFrame(erros_unificacao))


WARNING *** file size (442891) not 512 + multiple of sector size (512)
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
*** No CODEPAGE record, no encoding_override: will use 'iso-8859-1'
Arquivos processados: 4241
Arquivos com dados aproveitados: 4184
Linhas no dataframe final: 5583887
Arquivos com erro/sem cabeçalho: 57


,arquivo,etapa,erro
0,2018_01062018.xls,leitura,openpyxl: File is not a zip file | xlrd: | en...
1,2018_01092018.xls,leitura,openpyxl: File is not a zip file | xlrd: | en...
2,2018_03122018_Dados para Transparência.xls,leitura,openpyxl: File is not a zip file | xlrd: | en...
3,2018_06062018.xls,leitura,openpyxl: File is not a zip file | xlrd: | en...
4,2018_07062018.xls,leitura,openpyxl: File is not a zip file | xlrd: | en...
5,2018_08062018.xls,leitura,openpyxl: File is not a zip file | xlrd: | en...
6,2018_10112018_Dados para Transparência.xls,leitura,openpyxl: File is not a zip file | xlrd: | en...
7,2018_12112018_Dados para Transparência.xls,leitura,openpyxl: File is not a zip file | xlrd: | en...
8,2018_14092018.xls,leitura,openpyxl: File is not a zip file | xlrd: | en...
9,2018_15102018_Dados para Transparência.xls,leitura,openpyxl: File is not a zip file | xlrd: | en...


In [6]:
print(df_final_unificado.dtypes)

data                                   str
tipo                                   str
area                                   str
empresa                                str
linha                                  str
passageiros_pagantes_em_dinheiro       str
passageiros_comum_vt                   str
passageiros_bilhete_mensal             str
passageiro_estudante                   str
passageiro_estudante_mensal            str
passageiros_vale_transporte         object
passageiros_pagantes                object
passageiros_vt_mensal                  str
passageiros_integracao_metro        object
passageiros_integracao_onibus          str
passageiros_gratuidade                 str
passageiros_metro_mensal            object
total_pagantes                         str
ano_arquivo                          int64
arquivo_origem                         str
dtype: object


In [11]:
df_tipos_corrigidos = df_final_unificado.copy()
# única data que esta no padrão excel e precisa ser convertida para date normal
df_tipos_corrigidos.loc[df_tipos_corrigidos["data"].astype(str).str.strip() == "42065","data"] = "2015-03-02"

#
df_tipos_corrigidos["data"] = pd.to_datetime(df_tipos_corrigidos["data"], format="mixed").dt.normalize()
df_tipos_corrigidos["passageiros_pagantes_em_dinheiro"] = int(df_tipos_corrigidos["passageiros_pagantes_em_dinheiro"])


df_tipos_corrigidos.head(5)

TypeError: int() argument must be a string, a bytes-like object or a real number, not 'Series'

In [ ]:
# Salva em Parquet
arquivo_parquet = os.path.join(DADOS_DIR, "passageiros_unificado.parquet")
df_tipos_corrigidos.to_parquet(arquivo_parquet, index=False, engine="pyarrow")
print(f"Arquivo final: {os.path.abspath(arquivo_parquet)}")